In [ ]:
!rm -rf NLP_term_project
!git clone https://github.com/thit2003/NLP_term_project.git

Cloning into 'NLP_term_project'...
remote: Enumerating objects: 28110, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 28110 (delta 1), reused 12 (delta 1), pack-reused 28094 (from 1)
Receiving objects: 100% (28110/28110), 272.73 MiB | 11.81 MiB/s, done.
Resolving deltas: 100% (2987/2987), done.


In [ ]:
import os
import pandas as pd
import numpy as np
from datasets import Dataset, concatenate_datasets, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)

MODEL_NAME = "xlm-roberta-base"

LABEL2ID = {"positive": 0, "neutral": 1, "negative": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

In [ ]:
# -------------------------
# Twitter sentiment loaders
# -------------------------
def load_twitter_train(path="/content/NLP_term_project/twitter_sentiment/train.csv") -> Dataset:
    """
    train.csv has many columns.
    We only need: text + sentiment
    Handles non-UTF8 encodings robustly.
    """
    # Try common encodings
    last_err = None
    for enc in ("utf-8", "utf-8-sig", "cp1252", "latin1"):
        try:
            df = pd.read_csv(path, encoding=enc)
            break
        except UnicodeDecodeError as e:
            last_err = e
            df = None

    if df is None:
        raise last_err  # re-raise if all fail

    df = df[["text", "sentiment"]].rename(columns={"sentiment": "label"})
    df["text"] = df["text"].astype(str)
    df["label"] = df["label"].astype(str).str.strip().str.lower()
    df = df[df["label"].isin(LABEL2ID)].dropna(subset=["text", "label"])
    df["labels"] = df["label"].map(LABEL2ID).astype(int)
    return Dataset.from_pandas(df[["text", "labels"]])


def load_twitter_test(path="/content/NLP_term_project/twitter_sentiment/test.csv") -> Dataset:
    # test.csv is: <text>,<label> with no header; texts may contain commas
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line.strip() or "," not in line:
                continue
            text, label = line.rsplit(",", 1)
            text = text.strip()
            label = label.strip().lower()
            if text and label in LABEL2ID:
                rows.append({"text": text, "labels": LABEL2ID[label]})
    return Dataset.from_list(rows)


# -------------------------
# Wisesight loaders
# -------------------------
def load_wisesight_kaggle_comp(split: str) -> Dataset:
    """
    Uses files under:
      wisesight-sentiment/kaggle-competition/{train.txt, train_label.txt, test.txt, test_label.txt}
    Wisesight labels: pos/neu/neg/q
    We'll map q -> neutral.
    """
    base = "/content/NLP_term_project/wisesight-sentiment/kaggle-competition"
    if split == "train":
        text_path = os.path.join(base, "train.txt")
        label_path = os.path.join(base, "train_label.txt")
    elif split == "test":
        text_path = os.path.join(base, "test.txt")
        label_path = os.path.join(base, "test_label.txt")
    else:
        raise ValueError("split must be 'train' or 'test'")

    with open(text_path, "r", encoding="utf-8") as f:
        texts = [x.rstrip("\n") for x in f]
    with open(label_path, "r", encoding="utf-8") as f:
        labs = [x.rstrip("\n").strip().lower() for x in f]

    # Map to our 3 classes
    ws_map = {"pos": "positive", "neu": "neutral", "neg": "negative", "q": "neutral"}

    rows = []
    for t, l in zip(texts, labs):
        if not t:
            continue
        l2 = ws_map.get(l)
        if l2 is None:
            continue
        rows.append({"text": t, "labels": LABEL2ID[l2]})

    return Dataset.from_list(rows)


# -------------------------
# SST-2 loader
# -------------------------
def load_sst2(split: str) -> Dataset:
    """
    SST-2 labels:
      0 = negative
      1 = positive
    There is NO neutral class.
    We map:
      1 -> positive (0)
      0 -> negative (2)
    """
    sst2 = load_dataset("glue", "sst2")
    ds = sst2[split]
    rows = []
    for ex in ds:
        text = ex["sentence"]
        lab = ex["label"]
        if lab == 1:
            rows.append({"text": text, "labels": LABEL2ID["positive"]})
        elif lab == 0:
            rows.append({"text": text, "labels": LABEL2ID["negative"]})
    return Dataset.from_list(rows)

In [ ]:
# -------------------------
# Tokenization
# -------------------------
def tokenize_dataset(ds: Dataset, tokenizer, max_length=128) -> Dataset:
    def tok(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            padding="max_length",
            max_length=max_length,
        )

    ds = ds.map(tok, batched=True, desc="Tokenizing")
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    return ds

In [ ]:
# 1) Load Datasets
tw_train = load_twitter_train()
tw_eval = load_twitter_test()

ws_train = load_wisesight_kaggle_comp("train")
ws_eval = load_wisesight_kaggle_comp("test")

sst2_train = load_sst2("train")
sst2_eval = load_sst2("validation")

# 2) Combine train/eval
# Train = twitter_train + wisesight_train + sst2_train
train_ds = concatenate_datasets([tw_train, ws_train, sst2_train])

# Eval = twitter_test + wisesight_test + sst2_validation
eval_ds = concatenate_datasets([tw_eval, ws_eval, sst2_eval])

print("Dataset sizes:")
print(f"  twitter_train: {len(tw_train)}")
print(f"  wisesight_train: {len(ws_train)}")
print(f"  sst2_train: {len(sst2_train)}")
print(f"  TOTAL train: {len(train_ds)}")
print(f"  TOTAL eval:  {len(eval_ds)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

Dataset sizes:
  twitter_train: 27481
  wisesight_train: 24063
  sst2_train: 67349
  TOTAL train: 118893
  TOTAL eval:  7080


In [ ]:
# 3) Tokenize
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_dataset(ds, max_length=128):
    def tok(batch):
        return tokenizer(batch["text"], truncation=True, padding=False, max_length=max_length)

    ds = ds.map(tok, batched=True, desc="Tokenizing")
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    return ds

# 4) Build Model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = (preds == labels).mean()

    f1s = []
    for c in [0, 1, 2]:
        tp = np.sum((preds == c) & (labels == c))
        fp = np.sum((preds == c) & (labels != c))
        fn = np.sum((preds != c) & (labels == c))
        precision = tp / (tp + fp + 1e-12)
        recall = tp / (tp + fn + 1e-12)
        f1 = 2 * precision * recall / (precision + recall + 1e-12)
        f1s.append(f1)

    return {"accuracy": float(acc), "f1_macro": float(np.mean(f1s))}

# 5) Define Training Arguments
training_args = TrainingArguments(
    output_dir="./results_all",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none",
)

from transformers import DataCollatorWithPadding, Trainer

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
    processing_class=tokenizer,
)

In [ ]:
# Model Training
print("Setting up Trainer...")
trainer.train()
metrics = trainer.evaluate()
print("Final eval:", metrics)

# Save Model
out_dir = "./final_model_v2"

model.config.id2label = ID2LABEL
model.config.label2id = LABEL2ID

trainer.save_model(out_dir)
tokenizer.save_pretrained(out_dir)

print("Saved model to:", out_dir)
print("id2label:", model.config.id2label)
print("label2id:", model.config.label2id)

Setting up Trainer...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.383708,0.578229,0.774718,0.775058
2,0.285247,0.546461,0.791808,0.792672
3,0.257624,0.603771,0.793079,0.794512


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Final eval: {'eval_loss': 0.6037706732749939, 'eval_accuracy': 0.7930790960451978, 'eval_f1_macro': 0.7945124457189993, 'eval_runtime': 27.2135, 'eval_samples_per_second': 260.165, 'eval_steps_per_second': 8.158, 'epoch': 3.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model to: ./final_model_v2
id2label: {0: 'positive', 1: 'neutral', 2: 'negative'}
label2id: {'positive': 0, 'neutral': 1, 'negative': 2}


In [ ]:
import shutil

# Copy the entire folder to MyDrive
shutil.copytree('./final_model_v2', '/content/drive/MyDrive/final_model_v2')

'/content/drive/MyDrive/final_model_v2'